# Coherencia y Entrelazamiento como Recursos

Este notebook implementa las medidas de coherencia y entrelazamiento del módulo 22 del tutorial.

**Objetivos:**
- Calcular coherencia ℓ₁ y entropía relativa de coherencia para estados representativos
- Calcular entropía de entrelazamiento para estados de Bell, W y GHZ
- Visualizar cómo el canal de desfase destruye la coherencia manteniendo las poblaciones
- Verificar que un CNOT convierte coherencia en entrelazamiento

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, entropy, partial_trace

print('Qiskit importado correctamente')

## 1. Medidas de Coherencia

In [ ]:
def coherence_l1(rho: np.ndarray) -> float:
    """Coherencia ℓ₁: suma de |ρ_ij| para i≠j."""
    n = rho.shape[0]
    return sum(abs(rho[i, j]) for i in range(n) for j in range(n) if i != j)

def coherence_relative_entropy(rho: np.ndarray) -> float:
    """Coherencia entropía relativa: S(Δ(ρ)) - S(ρ)."""
    dm = DensityMatrix(rho)
    S_rho = entropy(dm, base=2)
    diagonal = np.diag(np.diag(rho))
    S_diag = entropy(DensityMatrix(diagonal), base=2)
    return float(S_diag - S_rho)

estados = {
    '|0⟩': np.array([[1, 0], [0, 0]], dtype=complex),
    '|1⟩': np.array([[0, 0], [0, 1]], dtype=complex),
    '|+⟩': np.array([[0.5, 0.5], [0.5, 0.5]], dtype=complex),
    '|-⟩': np.array([[0.5, -0.5], [-0.5, 0.5]], dtype=complex),
    'Mezcla 50/50': np.array([[0.5, 0], [0, 0.5]], dtype=complex),
}

print(f'{'Estado':<15} {'C_ℓ1':<10} {'C_rel':<10} {'S(ρ)'}')
print('-' * 45)
for name, rho in estados.items():
    c_l1 = coherence_l1(rho)
    c_rel = coherence_relative_entropy(rho)
    S = float(entropy(DensityMatrix(rho), base=2))
    print(f'{name:<15} {c_l1:<10.4f} {c_rel:<10.4f} {S:.4f}')

## 2. Canal de Desfase: Destrucción Selectiva de Coherencia

In [ ]:
def phase_damping_channel(rho: np.ndarray, p: float) -> np.ndarray:
    """Canal de desfase puro: K0 = diag(1, sqrt(1-p)), K1 = diag(0, sqrt(p))."""
    K0 = np.array([[1, 0], [0, np.sqrt(1-p)]])
    K1 = np.array([[0, 0], [0, np.sqrt(p)]])
    return K0 @ rho @ K0.conj().T + K1 @ rho @ K1.conj().T

rho_plus = np.array([[0.5, 0.5], [0.5, 0.5]], dtype=complex)
p_values = np.linspace(0, 1, 100)

coherences = []
populations_11 = []

for p in p_values:
    rho_out = phase_damping_channel(rho_plus, p)
    coherences.append(abs(rho_out[0, 1]))
    populations_11.append(rho_out[1, 1].real)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(p_values, coherences, color='#e74c3c', linewidth=2, label='|ρ₀₁| (coherencia)')
ax1.plot(p_values, [0.5*np.sqrt(1-p) for p in p_values], 'k--', alpha=0.5, label='Teoría: ½√(1-p)')
ax1.set_xlabel('Parámetro de desfase p')
ax1.set_ylabel('|ρ₀₁|')
ax1.set_title('Coherencia bajo canal de desfase')
ax1.legend()
ax1.set_ylim(0, 0.6)

ax2.plot(p_values, populations_11, color='#3498db', linewidth=2, label='ρ₁₁ (población)')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Valor constante 0.5')
ax2.set_xlabel('Parámetro de desfase p')
ax2.set_ylabel('ρ₁₁')
ax2.set_title('Población bajo canal de desfase')
ax2.legend()
ax2.set_ylim(0, 1)

plt.suptitle('Canal de desfase puro sobre |+⟩: coherencia vs población', fontsize=13)
plt.tight_layout()
plt.show()
print('Observación: el canal de desfase destruye la coherencia (ρ₀₁ → 0) sin cambiar las poblaciones (ρ₁₁ = 0.5 constante)')

## 3. Entropía de Entrelazamiento

In [ ]:
def entanglement_entropy(state_vector: np.ndarray, n_qubits: int, subsystem_A: list) -> float:
    """Entropía de von Neumann del estado reducido del subsistema A."""
    dm = DensityMatrix(state_vector)
    subsystem_B = [i for i in range(n_qubits) if i not in subsystem_A]
    rho_A = partial_trace(dm, subsystem_B)
    return float(entropy(rho_A, base=2))

# Estados de 2 qubits
bell_plus   = np.array([1, 0, 0, 1]) / np.sqrt(2)   # |Φ+>
bell_minus  = np.array([1, 0, 0, -1]) / np.sqrt(2)  # |Φ->
psi_plus    = np.array([0, 1, 1, 0]) / np.sqrt(2)   # |Ψ+>
producto    = np.array([1, 0, 0, 0], dtype=complex)   # |00>
producto_pp = np.array([1, 1, 1, 1]) / 2              # |++>

estados_2q = {
    '|Φ+⟩ (Bell)': bell_plus,
    '|Φ-⟩ (Bell)': bell_minus,
    '|Ψ+⟩ (Bell)': psi_plus,
    '|00⟩ (producto)': producto,
    '|++⟩ (producto)': producto_pp,
}

print('Estados de 2 qubits:')
print(f'{'Estado':<20} {'E (ebits)'}')
print('-' * 35)
for name, sv in estados_2q.items():
    E = entanglement_entropy(sv, 2, [0])
    print(f'{name:<20} {E:.4f}')

# Estado W y GHZ de 3 qubits
W   = np.array([0, 1, 1, 0, 1, 0, 0, 0]) / np.sqrt(3)
GHZ = np.array([1, 0, 0, 0, 0, 0, 0, 1]) / np.sqrt(2)

print('\nEstados de 3 qubits:')
for name, sv in [('|W⟩', W), ('|GHZ⟩', GHZ)]:
    for A in [[0], [1], [2], [0, 1]]:
        E = entanglement_entropy(sv, 3, A)
        print(f'  {name} subsistema {A}: E = {E:.4f} ebits')

## 4. CNOT convierte coherencia en entrelazamiento

In [ ]:
# Verificar: coh(|+>) = 1 ebit → E(|Φ+>) = 1 ebit tras CNOT
qc = QuantumCircuit(2)
qc.h(0)   # Coherencia ℓ₁ = 1.0 en el qubit 0

sv_before = Statevector.from_instruction(qc)
rho_before = DensityMatrix(sv_before)

# Estado reducido del qubit 0
rho_q0 = partial_trace(rho_before, [1]).data
coh_antes = coherence_l1(rho_q0)
E_antes = entanglement_entropy(sv_before.data, 2, [0])
print(f'ANTES del CNOT:')
print(f'  Coherencia ℓ₁ del qubit 0: {coh_antes:.4f}')
print(f'  Entrelazamiento: {E_antes:.4f} ebits')

qc.cx(0, 1)  # CNOT convierte coherencia en entrelazamiento
sv_after = Statevector.from_instruction(qc)
rho_after = DensityMatrix(sv_after)
rho_q0_after = partial_trace(rho_after, [1]).data
coh_despues = coherence_l1(rho_q0_after)
E_despues = entanglement_entropy(sv_after.data, 2, [0])

print(f'\nDESPUÉS del CNOT:')
print(f'  Coherencia ℓ₁ del qubit 0: {coh_despues:.4f}')
print(f'  Entrelazamiento: {E_despues:.4f} ebits')
print(f'\nEstado final: {sv_after}')
print(f'  → Es |Φ+⟩ = (|00⟩ + |11⟩)/√2 ✓')
print(f'\nConclusión: 1 ebit de coherencia → 1 ebit de entrelazamiento')